# BEV Calibration Check

SSH 포트포워딩으로 Jupyter를 열고, 두 CSI 카메라 중 BEV로 쓸 카메라만 골라 캡처한 뒤 `calib.json`과 확인 이미지를 만든다.

로컬 PC에서 접속 예시:

```bash
ssh -L 8888:localhost:8888 ircv16@<ROVER_IP>
```

로버 SSH 터미널에서 실행 예시:

```bash
cd /home/ircv16/team/final_project
jupyter notebook --no-browser --ip=127.0.0.1 --port=8888
```


## 1. Config

In [1]:
from pathlib import Path
import sys, time, subprocess

ROOT = Path('/home/ircv16/team')
PROJECT = ROOT / 'final_project'
CALIB_DIR = PROJECT / 'calib'
CALIB_DIR.mkdir(parents=True, exist_ok=True)

# 체커보드 내부 코너 수. 보드가 다르면 여기만 수정.
ROWS = 6
COLS = 9
SQUARE_M = 0.025

sys.path.insert(0, str(ROOT))
from calibration.camera import Camera

print('project:', PROJECT)
print('calib dir:', CALIB_DIR)

project: /home/ircv16/team/final_project
calib dir: /home/ircv16/team/final_project/calib


## 2. 두 카메라 프리뷰

`sensor_id=0`, `sensor_id=1`을 동시에 몇 초 동안 보여준다. 아래 화면에서 바닥/차선 쪽을 보는 카메라가 BEV 카메라다.

In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display

cams = {}
for sensor_id in (0, 1):
    try:
        cams[sensor_id] = Camera(sensor_id=sensor_id, capture_width=1280, capture_height=720, capture_fps=15)
        print(f'sensor_id={sensor_id}: opened')
    except Exception as e:
        print(f'sensor_id={sensor_id}: failed: {e}')

views = {sid: widgets.Image(format='jpeg', width=480) for sid in cams}
labels = [widgets.VBox([widgets.HTML(f'<b>sensor_id={sid}</b>'), view]) for sid, view in views.items()]
display(widgets.HBox(labels))

def rgb_to_jpeg_bytes(rgb):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode('.jpg', bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    if not ok:
        raise RuntimeError('cv2.imencode failed')
    return buf.tobytes()

def preview(seconds=20, fps=5):
    end = time.time() + seconds
    delay = 1.0 / fps
    while time.time() < end:
        for sid, cam in cams.items():
            frame = cam.read()
            if frame is not None:
                views[sid].value = rgb_to_jpeg_bytes(frame)
        time.sleep(delay)

preview(seconds=20, fps=5)

GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode  = 4 
   Output Stream W = 1280 H = 7

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1100) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success


[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   Camera index = 0 
   Camera mode  = 4 
   Output Stream W = 1280 H = 7

...

GST_ARGUS: Cleaning up
CONSUMER: Done Success
GST_ARGUS: Done Success
GST_ARGUS: Creating output stream
CONSUMER: Waiting until producer is connected...
GST_ARGUS: Available Sensor modes :
GST_ARGUS: 3280 x 2464 FR = 21.000000 fps Duration = 47619048 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 3280 x 1848 FR = 28.000001 fps Duration = 35714284 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1920 x 1080 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1640 x 1232 FR = 29.999999 fps Duration = 33333334 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: 1280 x 720 FR = 59.999999 fps Duration = 16666667 ; Analog Gain range min 1.000000, max 10.625000; Exposure Range min 13000, max 683709000;

GST_ARGUS: Running with following settings:
   

[ WARN:0] global ./modules/videoio/src/cap_gstreamer.cpp (1390) setProperty OpenCV | GStreamer warning: GStreamer: unhandled property


sensor_id=1: opened


(Argus) Error FileOperationFailed: Failed socket read: Connection reset by peer (in src/rpc/socket/common/SocketUtils.cpp, function readSocket(), line 79)
(Argus) Error FileOperationFailed: Unexpected error in reading socket (in src/rpc/socket/client/ClientSocketManager.cpp, function recvThreadCore(), line 277)
(Argus) Error FileOperationFailed: Receive worker failure, notifying 1 waiting threads (in src/rpc/socket/client/ClientSocketManager.cpp, function recvThreadCore(), line 350)
(Argus) Error InvalidState: Argus client is exiting with 1 outstanding client threads (in src/rpc/socket/client/ClientSocketManager.cpp, function recvThreadCore(), line 366)
(Argus) Error FileOperationFailed: Receiving thread terminated with error (in src/rpc/socket/client/ClientSocketManager.cpp, function recvThreadWrapper(), line 379)
(Argus) Error FileOperationFailed: Client thread received an error from socket (in src/rpc/socket/client/ClientSocketManager.cpp, function send(), line 145)
(Argus) Error Fi

CONSUMER: ERROR OCCURRED
CONSUMER: Done Success


## 3. BEV 카메라 선택 후 한 장 저장

위 프리뷰를 보고 `BEV_SENSOR_ID`를 `0` 또는 `1`로 고른다. 체커보드를 BEV 카메라 시야 안의 바닥에 놓고 실행.

In [ ]:
BEV_SENSOR_ID = 0  # 필요하면 1로 변경

from IPython.display import Image, display

cam = cams.get(BEV_SENSOR_ID)
if cam is None:
    cam = Camera(sensor_id=BEV_SENSOR_ID, capture_width=1280, capture_height=720, capture_fps=15)
    cams[BEV_SENSOR_ID] = cam

# 최신 프레임을 얻기 위해 몇 장 버림.
frame = None
for _ in range(10):
    frame = cam.read()
    time.sleep(0.05)

if frame is None:
    raise RuntimeError('No frame from selected BEV camera')

ts = time.strftime('%Y%m%d_%H%M%S')
capture_path = CALIB_DIR / f'bev_capture_sensor{BEV_SENSOR_ID}_{ts}.jpg'
cv2.imwrite(str(capture_path), cv2.cvtColor(frame, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
print('saved:', capture_path)
display(Image(filename=str(capture_path), width=640))

## 4. 캘리브레이션 생성 및 결과 확인

`calib_source.png`는 검출된 체커보드 영역, `calib_warped.png`는 BEV warp 결과다. 실패하면 `ROWS`, `COLS`, `SQUARE_M` 또는 체커보드 위치/초점을 확인.

In [ ]:
calib_json = CALIB_DIR / 'calib.json'
cmd = [
    sys.executable,
    str(PROJECT / 'data_pipeline' / 'bev_calibration.py'),
    '--image', str(capture_path),
    '--out', str(calib_json),
    '--rows', str(ROWS),
    '--cols', str(COLS),
    '--square_m', str(SQUARE_M),
]
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT), text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('calibration failed')

print('calib json:', calib_json)
display(Image(filename=str(CALIB_DIR / 'calib_source.png'), width=640))
display(Image(filename=str(CALIB_DIR / 'calib_warped.png'), width=360))

## 5. 정리

In [ ]:
for cam in cams.values():
    try:
        cam.stop()
    except Exception:
        pass
print('camera stopped')